# Feature borrowing: do the sample datamarts' features help?

This notebook tests two groups of their features as extra columns on our feature store, under a rule written before any comparison is run:

| Group | From | Columns |
|---|---|---|
| Loan-type counts | Sample 1 | 9: how many loans of each type in `Type_of_Loan` |
| Ratios | Samples 2 and 3 | 8: rebuilt on our cleaned columns (their own tables strip minus signs while parsing) |

Not tested:
- recodings (Payment_Behaviour split in two, age bands, log income): no information our one-hot and raw columns don't already hold
- monthly clickstream lags: single months drift more than the 3-month average chosen in notebook 02, and 20 features × 6 months is 120 columns to monitor
- debt/salary and EMI/salary ratios: already in the feature store

## 1. Decision rule (written before the comparison)

A feature group is adopted only if all three hold:

1. **It improves the model we would deploy for all the time periods.** The selected model (logistic regression or XGBoost, whichever has the higher OOT AUC) gains at least +0.005 OOT AUC at each of three train dates: 2024-01-01, 2024-03-01 and 2024-05-01.
2. **The gain is not chance.** At 2024-03-01, the 95% bootstrap interval of the gain excludes 0. One OOT window holds about 950 applications, so its AUC moves by about ±0.01 from sampling alone.
3. **It is stable.** Every new column has PSI below 0.1 against the training split in each month the model scores, 2024-03 to 2025-01.

Otherwise the feature store keeps its own features: every extra column is one more thing to monitor.

In [1]:
import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import xgboost as xgb

pd.set_option("display.width", 200)

In [2]:
# gold stores built by the DAG, and the silver financials that still hold Type_of_Loan
features = pd.read_parquet("../datamart/gold/feature_store/")
labels = pd.read_parquet("../datamart/gold/label_store/")
financials = pd.read_parquet("../datamart/silver/financials/", columns=["Customer_ID", "snapshot_date", "Type_of_Loan"])
for df in (features, financials, labels):
    df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])
labels["loan_start_date"] = pd.to_datetime(labels["loan_start_date"])
print("features", features.shape, "labels", labels.shape, "financials", financials.shape)

features (12500, 68) labels (12500, 6) financials (12500, 3)


## 2. Build the candidate columns

In [3]:
LOAN_TYPES = ["auto_loan", "credit-builder_loan", "debt_consolidation_loan", "home_equity_loan", "mortgage_loan",
              "not_specified", "payday_loan", "personal_loan", "student_loan"]

def loan_type_counts(type_of_loan):
    """'Auto Loan, Payday Loan, and Auto Loan' -> {'auto_loan': 2, 'payday_loan': 1}; blank means no loans"""
    if not isinstance(type_of_loan, str):
        return {}
    items = [x.strip().replace(" ", "_").lower() for x in type_of_loan.replace(" and ", ",").split(",")]
    return pd.Series([x for x in items if x]).value_counts().to_dict()

counts = financials["Type_of_Loan"].map(loan_type_counts)
print("types found:", sorted(set(k for c in counts for k in c)) == LOAN_TYPES)
loan_types = financials[["Customer_ID", "snapshot_date"]].copy()
for t in LOAN_TYPES:
    loan_types["lt_" + t] = counts.map(lambda c: c.get(t, 0))
lt_cols = ["lt_" + t for t in LOAN_TYPES]

def add_ratios(df):
    # ideas from Samples 2 and 3, computed on our cleaned columns; +1 keeps every denominator above 0
    df = df.copy()
    df["Loans_per_Credit_Item"] = df["Num_of_Loan"] / (df["Num_Bank_Accounts"] + df["Num_Credit_Card"] + 1)
    df["Num_Fin_Products"] = df["Num_Bank_Accounts"] + df["Num_Credit_Card"] + df["Num_of_Loan"]
    df["Repayment_Ability"] = df["Monthly_Inhand_Salary"] - df["Total_EMI_per_month"]
    df["Loan_Extent"] = df["Delay_from_due_date"] * df["Num_of_Loan"]
    df["Investment_Rate"] = df["Amount_invested_monthly"] / (df["Monthly_Inhand_Salary"] + 1)
    df["Balance_to_Debt"] = (df["Monthly_Balance"] + 1) / (df["Outstanding_Debt"] + 1)
    df["Inquiries_per_Loan"] = df["Num_Credit_Inquiries"] / (df["Num_of_Loan"] + 1)
    df["Has_Credit_Limit_Change"] = np.where(df["Changed_Credit_Limit"].isna(), np.nan, (df["Changed_Credit_Limit"] != 0).astype(float))
    return df

ratio_cols = ["Loans_per_Credit_Item", "Num_Fin_Products", "Repayment_Ability", "Loan_Extent", "Investment_Rate", "Balance_to_Debt", "Inquiries_per_Loan", "Has_Credit_Limit_Change"]
candidates = add_ratios(features).merge(loan_types, on=["Customer_ID", "snapshot_date"], how="left")
print("rows:", len(candidates), "of", len(features), "| applications without a loan-type row:", candidates["lt_auto_loan"].isna().sum())

types found: True
rows: 12500 of 12500 | applications without a loan-type row: 0


In [4]:
# checks: the counts add up to Num_of_Loan, no ratio is infinite, one applicant by hand
has_count = candidates["Num_of_Loan"].notna()
print("sum of counts == Num_of_Loan:", (candidates.loc[has_count, lt_cols].sum(axis=1) == candidates.loc[has_count, "Num_of_Loan"]).sum(), "of", has_count.sum())
print("infinite ratios:", np.isinf(candidates[ratio_cols]).sum().sum())
one = candidates[(candidates["Customer_ID"] == "CUS_0x10ee") & (candidates["snapshot_date"] == "2023-06-01")].iloc[0]
print("CUS_0x10ee 2023-06: Monthly_Balance", round(one["Monthly_Balance"], 2), "Outstanding_Debt", round(one["Outstanding_Debt"], 2), "-> Balance_to_Debt", round(one["Balance_to_Debt"], 4), "| by hand", round((one["Monthly_Balance"] + 1) / (one["Outstanding_Debt"] + 1), 4))
print("Type_of_Loan:", financials.loc[(financials["Customer_ID"] == "CUS_0x10ee") & (financials["snapshot_date"] == "2023-06-01"), "Type_of_Loan"].iloc[0], "| Num_of_Loan", one["Num_of_Loan"], "| counts", one[lt_cols].sum())

sum of counts == Num_of_Loan: 11933 of 11933
infinite ratios: 0
CUS_0x10ee 2023-06: Monthly_Balance 338.25 Outstanding_Debt 456.02 -> Balance_to_Debt 0.7423 | by hand 0.7423
Type_of_Loan: None | Num_of_Loan 0.0 | counts 0


**Findings:**
- The 9 counts add up to `Num_of_Loan` for every application that has it (11,933): they split a feature we already have by loan type. A blank `Type_of_Loan` is an applicant with no loans, so all zeros is right.
- No ratio is infinite, and CUS_0x10ee's `Balance_to_Debt` recomputed by hand matches.

## 3. Train and score, with and without each group

Same training as the pipeline will use: windows on the application month (6 months train/test, 2 months OOT, every label dated before the train date), rows sorted, a stratified 80/20 split, median imputation and scaling fitted on the training split, and a random search with 3-fold cross-validation on AUC.

In [5]:
# join features to labels on customer + application month, as in notebook 02
data = candidates.merge(
    labels[["Customer_ID", "loan_start_date", "label", "snapshot_date"]].rename(columns={"loan_start_date": "app_month", "snapshot_date": "label_date"}),
    left_on=["Customer_ID", "snapshot_date"], right_on=["Customer_ID", "app_month"], how="inner")
data = data.sort_values(["snapshot_date", "Customer_ID"]).reset_index(drop=True)
base_cols = [c for c in features.columns if c not in ("Customer_ID", "snapshot_date")]
feature_sets = {"base": base_cols, "+ loan types": base_cols + lt_cols, "+ ratios": base_cols + ratio_cols}

PARAM_DIST = {
    "logreg": {"C": np.logspace(-3, 2, 20)},
    "xgb": {"n_estimators": [25, 50, 100], "max_depth": [2, 3, 4], "learning_rate": [0.01, 0.1],
            "subsample": [0.6, 0.8], "colsample_bytree": [0.6, 0.8], "min_child_weight": [1, 3, 5]},
}

def train(cols, train_date, model_type):
    T = pd.Timestamp(train_date)
    oot_end = T - relativedelta(months=7)  # label known 6 months after application, and dated before T
    oot_start = oot_end - relativedelta(months=1)
    tt_start = oot_start - relativedelta(months=6)
    rows = data[(data["label_date"] < T) & (data["snapshot_date"] >= tt_start) & (data["snapshot_date"] <= oot_end)]
    oot = rows[rows["snapshot_date"] >= oot_start]
    train_test = rows[rows["snapshot_date"] < oot_start]
    X_train, X_test, y_train, y_test = train_test_split(train_test[cols], train_test["label"], test_size=0.2, random_state=88, shuffle=True, stratify=train_test["label"])
    imputer = SimpleImputer(strategy="median").fit(X_train)
    scaler = StandardScaler().fit(imputer.transform(X_train))
    prep = lambda X: scaler.transform(imputer.transform(X))
    estimator = LogisticRegression(max_iter=1000, random_state=88) if model_type == "logreg" else xgb.XGBClassifier(eval_metric="logloss", random_state=88)
    search = RandomizedSearchCV(estimator, PARAM_DIST[model_type], scoring="roc_auc", n_iter=20, cv=3, random_state=42, n_jobs=-1)
    model = search.fit(prep(X_train), y_train).best_estimator_
    oot_scores = model.predict_proba(prep(oot[cols]))[:, 1]
    return {"model": model, "cols": cols, "X_train": X_train, "y_oot": oot["label"].values, "oot_scores": oot_scores,
            "test": roc_auc_score(y_test, model.predict_proba(prep(X_test))[:, 1]), "oot": roc_auc_score(oot["label"], oot_scores)}

runs = {(fs, d, m): train(cols, d, m) for fs, cols in feature_sets.items()
        for d in ["2024-01-01", "2024-03-01", "2024-05-01"] for m in ["logreg", "xgb"]}
results = pd.DataFrame([{"features": fs, "train date": d, "model": m, "test AUC": r["test"], "OOT AUC": r["oot"]} for (fs, d, m), r in runs.items()])
results.pivot_table(index=["features", "model"], columns="train date", values="OOT AUC", sort=False).round(4)

train date           2024-01-01  2024-03-01  2024-05-01
features     model                                     
base         logreg      0.8308      0.8402      0.8456
             xgb         0.8461      0.8563      0.8470
+ loan types logreg      0.8268      0.8383      0.8434
             xgb         0.8467      0.8532      0.8434
+ ratios     logreg      0.8335      0.8453      0.8453
             xgb         0.8365      0.8477      0.8357

In [6]:
# the model we would deploy at each train date: the higher OOT AUC and the gain over base
selected = results.loc[results.groupby(["features", "train date"], sort=False)["OOT AUC"].idxmax()].set_index(["features", "train date"])
gain = selected["OOT AUC"].unstack("train date") - selected.loc["base", "OOT AUC"]
print(selected["model"].unstack("train date"), "\n")
print("gain of the selected model over base:")
gain.round(4)

train date   2024-01-01 2024-03-01 2024-05-01
features                                     
+ loan types        xgb        xgb        xgb
+ ratios            xgb        xgb     logreg
base                xgb        xgb        xgb 

gain of the selected model over base:


train date,2024-01-01,2024-03-01,2024-05-01
features,,,
+ loan types,0.0006,-0.0032,-0.0036
+ ratios,-0.0096,-0.0086,-0.0018
base,0.0000,0.0000,0.0000


**Findings:**
- XGBoost on our own features is selected at every train date (OOT AUC 0.846, 0.856, 0.847).
- Loan-type counts: the selected model's gain is +0.0006, −0.0032 and −0.0036. They don't help.
- Ratios: −0.0096, −0.0086 and −0.0018. They help logistic regression a little at 2024-03-01 (0.840 → 0.845), but logistic regression is not the model we would deploy, and XGBoost gets worse.

## 4. Is the gain more than chance?

Draw the OOT applications with replacement 1,000 times, compute both models' AUC on each draw, and take the middle 95% of the differences. Both models score the same applications, so the comparison is paired.

In [7]:
def bootstrap_gain(base_run, group_run, n=1000, seed=0):
    y, a, b = base_run["y_oot"], base_run["oot_scores"], group_run["oot_scores"]
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() != y[i].max():  # a draw needs both classes for an AUC
            diffs.append(roc_auc_score(y[i], b[i]) - roc_auc_score(y[i], a[i]))
    return np.percentile(diffs, [2.5, 97.5])

d = "2024-03-01"
base_run = runs[("base", d, selected.loc[("base", d), "model"])]
intervals = {}
for fs in ["+ loan types", "+ ratios"]:
    intervals[fs] = bootstrap_gain(base_run, runs[(fs, d, selected.loc[(fs, d), "model"])])
    print(f"{fs}: gain {gain.loc[fs, d]:+.4f}, 95% interval [{intervals[fs][0]:+.4f}, {intervals[fs][1]:+.4f}] on {len(base_run['y_oot'])} OOT applications")

+ loan types: gain -0.0032, 95% interval [-0.0096, +0.0026] on 952 OOT applications
+ ratios: gain -0.0086, 95% interval [-0.0177, +0.0003] on 952 OOT applications


+ ratios: gain -0.0086, 95% interval [-0.0177, +0.0003] on 952 OOT applications


**Findings:**
- Both intervals include 0: neither group's change at 2024-03-01 can be told apart from chance. A gain of 0.005 would itself be hard to see on one OOT window this size, which is why the rule also asks for it at every train date.

## 5. Are the new columns stable, and does the model use them?

In [8]:
def psi(reference, current):
    """Population Stability Index of current values against reference values.
    Bins are the reference deciles; missing values are their own bin, so a feed that stops shows up."""
    reference, current = pd.Series(reference, dtype=float), pd.Series(current, dtype=float)
    edges = np.unique(np.nanquantile(reference, np.linspace(0.1, 0.9, 9)))

    def proportions(values):
        counts = np.bincount(np.digitize(values.dropna(), edges, right=True), minlength=len(edges) + 1)
        counts = np.append(counts, values.isna().sum())
        return np.clip(counts / len(values), 1e-4, None)  # an empty bin would make log() infinite

    ref, cur = proportions(reference), proportions(current)
    return float(np.sum((cur - ref) * np.log(cur / ref)))

scored_months = pd.date_range("2024-03-01", "2025-01-01", freq="MS")
stability = []
for fs, cols in [("+ loan types", lt_cols), ("+ ratios", ratio_cols)]:
    reference = runs[(fs, "2024-03-01", "xgb")]["X_train"]
    xgb_run = runs[(fs, "2024-03-01", "xgb")]
    importance = pd.Series(xgb_run["model"].feature_importances_, index=xgb_run["cols"])
    for c in cols:
        stability.append({"group": fs, "column": c, "importance (xgb)": importance[c],
                          "worst PSI 2024-03..2025-01": max(psi(reference[c], candidates.loc[candidates["snapshot_date"] == m, c]) for m in scored_months)})
stability = pd.DataFrame(stability)
print(stability.groupby("group", sort=False).agg(importance_share=("importance (xgb)", "sum"), worst_psi=("worst PSI 2024-03..2025-01", "max")).round(3), "\n")
stability.round(3)

              importance_share  worst_psi
group                                    
+ loan types             0.014      0.026
+ ratios                 0.107      0.063 



,group,column,importance (xgb),worst PSI 2024-03..2025-01
0,+ loan types,lt_auto_loan,0.000,0.017
1,+ loan types,lt_credit-builder_loan,0.000,0.020
2,+ loan types,lt_debt_consolidation_loan,0.000,0.009
3,+ loan types,lt_home_equity_loan,0.000,0.014
4,+ loan types,lt_mortgage_loan,0.000,0.011
5,+ loan types,lt_not_specified,0.000,0.013
6,+ loan types,lt_payday_loan,0.000,0.013
7,+ loan types,lt_personal_loan,0.000,0.026
8,+ loan types,lt_student_loan,0.014,0.012
9,+ ratios,Loans_per_Credit_Item,0.014,0.041


**Findings:**
- Both groups are stable (worst PSI 0.026 for the counts, 0.063 for the ratios), so stability is not what decides.
- XGBoost barely uses the loan-type counts (1.4% of its importance, all on `lt_student_loan`). It gives the ratios 10.7%, taken from the raw columns they are built from, without a better OOT AUC.

## 6. Decision

In [9]:
for fs, cols in [("+ loan types", lt_cols), ("+ ratios", ratio_cols)]:
    every_date = bool((gain.loc[fs] >= 0.005).all())
    not_chance = bool(intervals[fs][0] > 0)
    stable = bool((stability.loc[stability["group"] == fs, "worst PSI 2024-03..2025-01"] < 0.1).all())
    print(f"{fs}: +0.005 at every date {every_date} | interval excludes 0 {not_chance} | PSI < 0.1 {stable} -> {'ADOPT' if every_date and not_chance and stable else 'DO NOT BORROW'}")

+ loan types: +0.005 at every date False | interval excludes 0 False | PSI < 0.1 True -> DO NOT BORROW
+ ratios: +0.005 at every date False | interval excludes 0 False | PSI < 0.1 True -> DO NOT BORROW


**Findings:**
- Neither group meets the rule: both are stable, but neither improves the deployed model at any train date beyond chance. The feature store keeps its own 66 features.
- For the deck: we tested Sample 1's loan-type counts and the ratio features of Samples 2 and 3 as extra columns; neither improved out-of-time AUC at three train dates (best gain +0.0006), so we kept our own features, which also keeps the number of features to monitor down.